In [1]:
import pandas as pd
import json
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem.porter import PorterStemmer

In [2]:
movies = pd.read_csv("../dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("../dataset/tmdb_5000_credits.csv")

In [3]:
new_movies=movies.merge(credits,on="title")

In [4]:
new_movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [5]:
new_movies=new_movies[['id','title','genres','keywords','overview','cast','crew']]
new_movies.head(1)

,id,title,genres,keywords,overview,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [6]:
new_movies.dropna(inplace=True)

In [7]:
def formatgenre(text):
    genres=[]
    for i in json.loads(text):
        genres.append(i["name"])
    return genres

In [8]:
new_movies['genres']=new_movies['genres'].apply(formatgenre)

In [9]:
def formatkeywords(text):
    keywords=[]
    for i in json.loads(text):
        keywords.append(i["name"])
    return keywords

In [10]:
new_movies['keywords']=new_movies['keywords'].apply(formatkeywords)

In [11]:
def castformatter(text):
    cast=[]
    z=0
    for i in json.loads(text):
        if z==3:
            break
        z=z+1
        cast.append(i["name"])
    return cast

In [12]:
new_movies['cast']=new_movies['cast'].apply(castformatter)

In [13]:
def crewformatter(text):
    director=[]
    for i in json.loads(text):
        if i["job"]=="Director":
            director.append(i["name"])
            break
    return director

In [14]:
new_movies['crew']=new_movies['crew'].apply(crewformatter)

In [15]:
new_movies["keywords"][0]

['culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d']

In [16]:
new_movies["overview"]=new_movies["overview"].apply(lambda x: x.split())
new_movies['crew']=new_movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
new_movies['genres']=new_movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
new_movies['cast']=new_movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
new_movies['keywords']=new_movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
new_movies["allkeywords"]=new_movies["overview"]+new_movies["keywords"]+new_movies["genres"]+new_movies["cast"]+new_movies["crew"]

In [17]:
movies=new_movies[["id","title","allkeywords"]]

In [18]:
movies["allkeywords"]=movies["allkeywords"].apply(lambda x:" ".join(x))

C:\Users\anish\AppData\Local\Temp\ipykernel_6912\2329735072.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies["allkeywords"]=movies["allkeywords"].apply(lambda x:" ".join(x))


In [19]:
movies.head()

,id,title,allkeywords
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [20]:
movies["allkeywords"]=movies["allkeywords"].apply(lambda x:x.lower())

C:\Users\anish\AppData\Local\Temp\ipykernel_6912\3099174844.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies["allkeywords"]=movies["allkeywords"].apply(lambda x:x.lower())


In [21]:
movies["allkeywords"][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d action adventure fantasy sciencefiction samworthington zoesaldana sigourneyweaver jamescameron'

In [22]:
cv=CountVectorizer(max_features=5000,stop_words="english")

In [23]:
ps=PorterStemmer()

In [24]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [25]:
movies["allkeywords"]=movies["allkeywords"].apply(stem)

C:\Users\anish\AppData\Local\Temp\ipykernel_6912\3054010685.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies["allkeywords"]=movies["allkeywords"].apply(stem)


In [26]:
vectors=cv.fit_transform(movies["allkeywords"]).toarray()

In [27]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [28]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [29]:
similarity=cosine_similarity(vectors)

In [30]:
def recommend(movie):
    movie = movie.lower()
    if movie not in movies["title"].str.lower().values:
        print(f"'{movie}' not found. Did you mean one of these?")
        suggestions = movies[movies['title'].str.contains(movie.split()[0], case=False)]['title'].values[:5]
        print(suggestions)
        return
    movie_index = movies[movies["title"].str.lower() == movie].index[0]
    distances = similarity[movie_index]
    top_5 = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    for i in top_5:
        print(movies.iloc[i[0]].title)

In [31]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [32]:
import pickle

# Save movies DataFrame
with open("movies.pkl", "wb") as f:
    pickle.dump(movies, f)

# Save similarity matrix
with open("similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

# Save CountVectorizer (optional)
with open("cv.pkl", "wb") as f:
    pickle.dump(cv, f)

print("Model exported as pickle files successfully!")


Model exported as pickle files successfully!
